In [ ]:
from openai import OpenAI
from rich.console import Console
import json
import gradio as gr

In [ ]:
ollama_base_url = "http://localhost:11434/v1"
ollama_api_key = "ollama"
ollama_client = OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)

In [ ]:
todos = []
completed_tasks = []
console = Console()

def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed_tasks[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    console.print(result)
    return result

def create_todos(description: list[str]) -> str:
        todos.extend(description)
        completed_tasks.extend([False] * len(description))
        return get_todo_report()

create_todos_json = {
    "name": "create_todos",
    "description": "Create a list of todos based on the provided description.",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of todo descriptions to be created."
            }
        },
        "required": ["description"]
    }
}

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed_tasks[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark a todo as complete based on its index and provide completion notes.",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The index of the todo to be marked as complete (1-based index)."
            },
            "completion_notes": {
                "type": "string",         
                "description": "Notes to be displayed upon marking the todo as complete."
            }
        },
        "required": ["index", "completion_notes"]
    }
}

In [ ]:
def setup_slack_access():
    # Placeholder for Slack access setup
    pass

setup_slack_json = {
    "name": "setup_slack_access",
    "description": (
        "Use this tool to set up Slack access for a new student. "
        "Every student should have Slack access to communicate with teachers and classmates."
    )
}

In [ ]:
def setup_email_access():
    # Placeholder for email access setup
    pass

setup_email_json = {
    "name": "setup_email_access",
    "description": (
        "Use this tool to set up email access for a new student. "
        "Every student should have an email account for official school communication and class alerts."
    )
}

In [ ]:
def setup_documentation_access():
    # Placeholder for documentation access setup
    pass

setup_documentation_json = {
    "name": "setup_documentation_access",
    "description": (
        "Use this tool to set up documentation and school portal access for a new student. "
        "This gives them access to their class materials, syllabus, library resources, and school rules."
    )
}

In [ ]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json},
    {"type": "function", "function": setup_slack_json},
    {"type": "function", "function": setup_email_json},
    {"type": "function", "function": setup_documentation_json}
]

In [ ]:
system_message = """
    You are an onboarding assistant for new school children at a school. Your role is to help set up necessary tools and resources for the students based on their grade and classes.
    You will be provided with a list of tools and their descriptions, and you should use this information to determine which tools to set up for each student.
    When a new student is onboarded, you will receive a message with their grade and classes.
    Based on this information, you should determine which tools they need access to and use the appropriate tool from the list to set up their access.
    If you are unsure about which tools to set up, you can ask for more information about the student.
    Your goal is to ensure that students have access to all the necessary tools and resources (like Slack, Email, Documentation) they need for their classes.
    Always make use of your create todo tool to plan out the steps you would take to set up the new student from start to finish
    When a particular step is done, make use of the mark complete tool to notify the user that a step is now complete
    """

In [ ]:
def chat(message, history):
    system_message = """
    You are an onboarding assistant for new school children at a school. Your role is to help set up necessary tools and resources for the students based on their grade and classes.
    You will be provided with a list of tools and their descriptions, and you should use this information to determine which tools to set up for each student.
    When a new student is onboarded, you will receive a message with their grade and classes.
    Based on this information, you should determine which tools they need access to and use the appropriate tool from the list to set up their access.
    If you are unsure about which tools to set up, you can ask for more information about the student.
    Your goal is to ensure that students have access to all the necessary tools and resources (like Slack, Email, Documentation) they need for their classes.
    Always make use of your create todo tool to plan out the steps you would take to set up the new student from start to finish
    When a particular step is done, make use of the mark complete tool to notify the user that a step is now complete
    """

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}] 
    done = False
    while not done:
        response = ollama_client.chat.completions.create(
            model="gpt-oss:120b-cloud",
            messages=messages,
            tools=tools,
        )

        finish_reason = response.choices[0].finish_reason
             
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

def handle_tool_calls(tool_calls):
        results = []
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            print(f"Tool called: {tool_name}", flush=True)
            tool = globals().get(tool_name)
            result = tool(**arguments) if tool else {}
            results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
        return results

In [ ]:
ONBOARDING_WELCOME_MESSAGE = """
Welcome to the school, we are excited to have you onboard, and we look forward to the wonderful school year ahead!
I am Marvin, your AI friend and helper. I can help set up your student accounts and grant you access to the resources you need for your classes.
To get started, please introduce yourself. (Your name, your grade, your classes, and if you don't mind, your favorite subject or hobby!)
"""

chatbot = gr.Chatbot(value=[{"role": "assistant", "content": ONBOARDING_WELCOME_MESSAGE}], type="messages", height=750,)
gr.ChatInterface(chat, chatbot=chatbot, type="messages").launch(inbrowser=True)